<div style="background:linear-gradient(135deg,#1e1b4b 0%,#4338ca 55%,#6366f1 100%);border-radius:18px;padding:30px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#c7d2fe;font-weight:700;text-transform:uppercase">Chapter 107 · Solutions</div>
  <div style="font-size:30px;font-weight:900;line-height:1.1;margin:10px 0 6px">Core Classification &amp; Regression Algorithms &#183; Solutions</div>
  <div style="font-size:14px;color:#e0e7ff;max-width:740px;line-height:1.6">Five challenges, each verified in code.</div>

</div>

# Chapter 107 &#183; Practice Challenges, Solved
Five exercises on the core algorithms, worked in `scikit-learn` on the Chapter 107 heart-disease table.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns   # seaborn = high-level statistical plots (heatmaps, pairplots, count/bar plots)
from matplotlib.colors import ListedColormap
EM="#4338ca"; DEEP="#3730a3"; LIGHT="#c7d2fe"; INK="#1a2138"; GRID="#e6e9f2"; RED="#ef4444"; AMBER="#d97706"; GREEN="#059669"; BLUE="#2563eb"; PUR="#9333ea"; GREY="#94a3b8"; SLATE="#475569"; ORG="#4338ca"; CYAN="#0891b2"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","figure.dpi":110,"font.size":11,
   "axes.edgecolor":GRID,"axes.grid":True,"grid.color":GRID,"axes.axisbelow":True,"axes.spines.top":False,
   "axes.spines.right":False,"axes.titlesize":12,"axes.titleweight":"bold","legend.frameon":False})
sns.set_style("whitegrid")
BASE="https://raw.githubusercontent.com/johnfisher-ai/Statistics-Data-Science-AI-Visual-Book/main/data/"

In [2]:
import warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score
try: df = pd.read_excel('../../data/ch107_patients.xlsx', sheet_name='Data')
except FileNotFoundError: df = pd.read_excel(BASE + 'ch107_patients.xlsx', sheet_name='Data')
feat = ['age','resting_bp','cholesterol','max_hr','glucose','bmi']
X, y = df[feat], df['heart_disease']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print('loaded', df.shape)

loaded (700, 8)


<div style="background:#eef2ff;border-left:5px solid #4338ca;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#3730a3;letter-spacing:1px">CHALLENGE 1</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Train a KNN classifier</div>
<div style="color:#4a5578;margin-top:5px">Scale the features, fit KNN (k=15), and report test accuracy.</div>
</div>

In [3]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(15)).fit(X_tr, y_tr)
print(f'test accuracy = {accuracy_score(y_te, knn.predict(X_te)):.3f}')

test accuracy = 0.766


**Solution.** KNN measures distance, so the pipeline standardizes first; accuracy is scored on the held-out patients.

<div style="background:#eef2ff;border-left:5px solid #4338ca;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#3730a3;letter-spacing:1px">CHALLENGE 2</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Read a tree's importances</div>
<div style="color:#4a5578;margin-top:5px">Fit a shallow decision tree and print its most important feature.</div>
</div>

In [4]:
tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr, y_tr)
imp = pd.Series(tree.feature_importances_, index=feat).sort_values(ascending=False)
print(imp.round(3).to_string()); print('top feature:', imp.idxmax())

age            0.415
bmi            0.199
max_hr         0.139
glucose        0.120
resting_bp     0.071
cholesterol    0.056
top feature: age


**Solution.** Age carries the strongest splits, which matches medical intuition for heart-disease risk.

<div style="background:#eef2ff;border-left:5px solid #4338ca;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#3730a3;letter-spacing:1px">CHALLENGE 3</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">Head-to-head</div>
<div style="color:#4a5578;margin-top:5px">Compare logistic regression, a tree, and KNN by 5-fold cross-validated accuracy.</div>
</div>

In [5]:
models = {'Logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          'Tree': DecisionTreeClassifier(max_depth=4, random_state=0),
          'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier(15))}
for name,m in models.items(): print(f'{name:10s} {cross_val_score(m, X, y, cv=5).mean():.3f}')

Logistic   0.773
Tree       0.683
KNN        0.750


**Solution.** No model wins everywhere; cross-validate a few and let the numbers plus your constraints decide.

<div style="background:#eef2ff;border-left:5px solid #4338ca;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#3730a3;letter-spacing:1px">CHALLENGE 4</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">SVM: the kernel matters</div>
<div style="color:#4a5578;margin-top:5px">Compare a linear SVM and an RBF SVM by cross-validated accuracy.</div>
</div>

In [6]:
for k in ['linear','rbf']:
    acc = cross_val_score(make_pipeline(StandardScaler(), SVC(kernel=k)), X, y, cv=5).mean()
    print(f'SVM ({k:6s}) accuracy = {acc:.3f}')

SVM (linear) accuracy = 0.761
SVM (rbf   ) accuracy = 0.766


**Solution.** The kernel sets the boundary shape: linear draws a straight divide, RBF bends to the data; try both.

<div style="background:#eef2ff;border-left:5px solid #4338ca;border-radius:10px;padding:14px 18px;font-family:Inter,sans-serif">
<span style="font-size:12px;font-weight:800;color:#3730a3;letter-spacing:1px">CHALLENGE 5</span>
<div style="font-size:18px;font-weight:800;color:#1a2138;margin-top:3px">The regression twin</div>
<div style="color:#4a5578;margin-top:5px">Predict max heart rate (a number) with a KNN regressor and linear regression; compare R-squared.</div>
</div>

In [7]:
Xr = df[['age','resting_bp','cholesterol','glucose','bmi']]; yr = df['max_hr']
for name,r in {'KNN reg': make_pipeline(StandardScaler(), KNeighborsRegressor(15)), 'Linear reg': LinearRegression()}.items():
    print(f'{name:10s} R2 = {cross_val_score(r, Xr, yr, cv=5, scoring="r2").mean():.3f}')

KNN reg    R2 = 0.274
Linear reg R2 = 0.356


**Solution.** Each classifier has a regressor twin using the same idea; here they predict a number instead of a class.

---
<div style="text-align:center;color:#8b94b3;font-size:12px;margin-top:10px">Statistics, Data Science and AI: A Visual Handbook · © 2026 John Fisher</div>